In [1]:
import re
import torch
import numpy as np
import pandas as pd
import pickle

In [2]:
# import re
# import numpy as np

# # =========================
# # NORMALIZE
# # =========================
# def normalize_code(code):
#     if not isinstance(code, str):
#         return ""
#     code = code.replace("\\r", "\n").replace("\r", "\n")
#     code = re.sub(r'\s+', ' ', code)
#     return code


# # =========================
# # SPLIT FUNCTIONS
# # =========================
# def split_functions(code):
#     functions = []
#     pattern = re.compile(r'function\b.*?\{', re.DOTALL)

#     for match in pattern.finditer(code):
#         start = match.start()
#         brace_count = 0

#         for i in range(match.end() - 1, len(code)):
#             if code[i] == "{":
#                 brace_count += 1
#             elif code[i] == "}":
#                 brace_count -= 1
#                 if brace_count == 0:
#                     functions.append(code[start:i + 1])
#                     break

#     return functions


# # =========================
# # SPLIT BLOCKS → TREE
# # =========================
# def split_blocks(function_code):
#     code = normalize_code(function_code)

#     tokens = re.split(r'(;|\{|\})', code)

#     stmts = []
#     current = ""

#     for t in tokens:
#         current += t
#         if t in [";", "{", "}"]:
#             stmts.append(current.strip())
#             current = ""

#     if current.strip():
#         stmts.append(current.strip())

#     stack = []
#     root = []
#     current_list = root

#     def classify(stmt):
#         if re.search(r'\bif\b', stmt):
#             return "IF"
#         elif re.search(r'\belse\b', stmt):
#             return "ELSE"
#         elif re.search(r'\b(for|while)\b', stmt):
#             return "LOOP"
#         elif re.search(r'\breturn\b', stmt):
#             return "RETURN"
#         else:
#             return "NORMAL"

#     for stmt in stmts:
#         stmt = stmt.strip()
#         if not stmt:
#             continue

#         node = {
#             "type": classify(stmt),
#             "text": stmt,
#             "children": []
#         }

#         if stmt.endswith("{"):
#             current_list.append(node)
#             stack.append(current_list)
#             current_list = node["children"]

#         elif stmt == "}":
#             if stack:
#                 current_list = stack.pop()

#         else:
#             current_list.append(node)

#     return root


# # =========================
# # ADVANCED FEATURE
# # =========================
# #42 chieu
# # # =========================
# # # ADVANCED FEATURE (CẬP NHẬT MỚI - THÊM NHIỀU FEATURES VỀ VULNERABILITY)
# # # =========================
# def extract_features(block):
#     """
#     Phiên bản nâng cấp: thêm ~15 features mới tập trung vào:
#     - Reentrancy risk (external call trước state update)
#     - Assembly / low-level code
#     - Event emission
#     - Access control (msg.sender trong require/if)
#     - Crypto operations (keccak, ecrecover...)
#     - Storage/memory/calldata
#     - Mapping/Array write
#     - Timestamp/Block dependency bổ sung
#     - Balance check
#     - Payable / fallback
#     """
#     text = block["text"].lower()
#     t = block["type"]

#     tokens = re.findall(r'\w+', text)
#     unique_tokens = len(set(tokens))

#     # ========================
#     # FEATURES CŨ (GIỮ NGUYÊN + CẢI TIẾN NHẸ)
#     # ========================
#     num_calls = len(re.findall(r'\b(call|delegatecall|callcode|staticcall)\b', text))
#     num_transfer = len(re.findall(r'\b(transfer|send)\b', text))
#     num_require = len(re.findall(r'\b(require|assert|revert|throw)\b', text))

#     has_delegatecall = int("delegatecall" in text)
#     has_selfdestruct = int("selfdestruct" in text or "suicide" in text)
#     has_tx_origin = int("tx.origin" in text)

#     has_block_timestamp = int("block.timestamp" in text or "now" in text)
#     has_msg_sender = int("msg.sender" in text)
#     has_msg_value = int("msg.value" in text)

#     num_arith = len(re.findall(r'[\+\-\*/%]', text))          # thêm %
#     num_compare = len(re.findall(r'(==|!=|>|<|>=|<=)', text))

#     has_state_write = int(bool(re.search(r'\b\w+\s*=\s*(?!=)', text)))
#     has_mapping_access = int(bool(re.search(r'\w+\s*\[.*?\]', text)))
#     has_external_call = int(num_calls > 0 or num_transfer > 0)

#     num_children = len(block["children"])

#     # ========================
#     # FEATURES MỚI (VULNERABILITY-SPECIFIC)
#     # ========================
#     has_assembly = int("assembly" in text)
#     num_emits = len(re.findall(r'\bemit\b', text))
#     has_payable = int("payable" in text)
#     has_balance = len(re.findall(r'\.balance', text))

#     has_block_number = int("block.number" in text)
#     has_ecrecover = int("ecrecover" in text)
#     has_keccak256 = int("keccak256" in text)
#     has_sha256 = int("sha256" in text)

#     has_storage = int("storage" in text)
#     has_memory = int("memory" in text)
#     has_calldata = int("calldata" in text)

#     num_mapping_writes = len(re.findall(r'\w+\s*\[.*?\]\s*=', text))
#     num_array_writes = len(re.findall(r'\w+\s*\[[^\]]+\]\s*=', text))

#     # Access control patterns
#     has_msg_sender_check = int(bool(
#         re.search(r'(require|assert|if).*?msg\.sender|msg\.sender.*?(require|assert|if|==|!=)', text)
#     ))

#     # Reentrancy heuristic (external call → state write trong cùng block)
#     has_reentrancy_risk = int(bool(
#         re.search(r'(call|send|transfer|delegatecall|staticcall).*?=', text, re.DOTALL)
#     ))

#     # Số lượng assignment (khác với ==)
#     num_assign = len(re.findall(r'(?<!\=)\=(?!\=)', text))

#     # ========================
#     # TRẢ VỀ VECTOR (tăng từ 27 → 42 features)
#     # ========================
#     return np.array([
#         # TYPE (4)
#         int(t == "IF"),
#         int(t == "ELSE"),
#         int(t == "LOOP"),
#         int(t == "RETURN"),

#         # CALL / SECURITY cũ (12)
#         int(num_calls > 0),
#         int(num_transfer > 0),
#         int(num_require > 0),
#         has_delegatecall,
#         has_selfdestruct,
#         has_tx_origin,
#         has_block_timestamp,
#         has_msg_sender,
#         has_msg_value,

#         # STATE / EXTERNAL cũ (3)
#         has_state_write,
#         has_mapping_access,
#         has_external_call,

#         # MATH cũ (2)
#         int(num_arith > 0),
#         int(num_compare > 0),

#         # STRUCTURE + SIZE cũ (5)
#         num_children,
#         len(tokens),
#         unique_tokens,
#         len(text),

#         # COUNT cũ (5)
#         num_calls,
#         num_transfer,
#         num_require,
#         num_arith,
#         num_compare,

#         # ==================== FEATURES MỚI ====================
#         has_assembly,
#         num_emits,
#         has_payable,
#         has_balance,
#         has_block_number,
#         has_ecrecover,
#         has_keccak256,
#         has_sha256,
#         has_storage,
#         has_memory,
#         has_calldata,
#         num_mapping_writes,
#         num_array_writes,
#         int(has_msg_sender_check),
#         int(has_reentrancy_risk),
#         num_assign,
#     ], dtype=np.float32)

# # def extract_features(block):
# #     text = block["text"].lower()
# #     t = block["type"]

# #     tokens = re.findall(r'\w+', text)
# #     unique_tokens = len(set(tokens))

# #     # ===== SMART CONTRACT =====
# #     num_calls = len(re.findall(r'\b(call|delegatecall|callcode)\b', text))
# #     num_transfer = len(re.findall(r'\b(transfer|send)\b', text))
# #     num_require = len(re.findall(r'\b(require|assert|revert|throw)\b', text))

# #     has_delegatecall = int("delegatecall" in text)
# #     has_selfdestruct = int("selfdestruct" in text)
# #     has_tx_origin = int("tx.origin" in text)

# #     has_block_timestamp = int("block.timestamp" in text)
# #     has_msg_sender = int("msg.sender" in text)
# #     has_msg_value = int("msg.value" in text)

# #     # ===== ARITHMETIC / LOGIC =====
# #     num_arith = len(re.findall(r'[\+\-\*/]', text))
# #     num_compare = len(re.findall(r'(==|!=|>|<|>=|<=)', text))

# #     # ===== STATE =====
# #     has_state_write = int(bool(re.search(r'\b\w+\s*=\s*(?!=)', text)))
# #     has_mapping_access = int(bool(re.search(r'\w+\s*\[.*?\]', text)))

# #     # ===== EXTERNAL CALL =====
# #     has_external_call = int(num_calls > 0 or num_transfer > 0)

# #     # ===== CONTROL =====
# #     num_children = len(block["children"])

# #     return np.array([
# #         # ===== TYPE =====
# #         int(t == "IF"),
# #         int(t == "ELSE"),
# #         int(t == "LOOP"),
# #         int(t == "RETURN"),

# #         # ===== CALL / SECURITY =====
# #         int(num_calls > 0),
# #         int(num_transfer > 0),
# #         int(num_require > 0),

# #         has_delegatecall,
# #         has_selfdestruct,
# #         has_tx_origin,

# #         has_block_timestamp,
# #         has_msg_sender,
# #         has_msg_value,

# #         # ===== STATE =====
# #         has_state_write,
# #         has_mapping_access,
# #         has_external_call,

# #         # ===== MATH =====
# #         int(num_arith > 0),
# #         int(num_compare > 0),

# #         # ===== STRUCTURE =====
# #         num_children,

# #         # ===== SIZE =====
# #         len(tokens),
# #         unique_tokens,
# #         len(text),

# #         # ===== COUNT =====
# #         num_calls,
# #         num_transfer,
# #         num_require,
# #         num_arith,
# #         num_compare

# #     ], dtype=np.float32)


# # =========================
# # BUILD CFG
# # =========================
# def build_cfg_from_tree(blocks, nodes, edges):
#     prev_idx = None
#     i = 0

#     while i < len(blocks):
#         block = blocks[i]

#         idx = len(nodes)
#         nodes.append(block)

#         # ===== SEQUENTIAL =====
#         if prev_idx is not None:
#             prev_type = nodes[prev_idx]["type"]
#             if prev_type not in ["RETURN", "IF"]:
#                 edges.append((prev_idx, idx))

#         # ===== IF =====
#         if block["type"] == "IF":

#             true_start = len(nodes)
#             build_cfg_from_tree(block["children"], nodes, edges)
#             true_end = len(nodes) - 1

#             if i + 1 < len(blocks) and blocks[i + 1]["type"] == "ELSE":
#                 else_block = blocks[i + 1]

#                 false_start = len(nodes)
#                 build_cfg_from_tree(else_block["children"], nodes, edges)
#                 false_end = len(nodes) - 1

#                 edges.append((idx, true_start))
#                 edges.append((idx, false_start))

#                 merge_idx = len(nodes)
#                 edges.append((true_end, merge_idx))
#                 edges.append((false_end, merge_idx))

#                 i += 1
#             else:
#                 edges.append((idx, true_start))

#         # ===== LOOP =====
#         elif block["type"] == "LOOP":
#             body_start = len(nodes)
#             build_cfg_from_tree(block["children"], nodes, edges)
#             body_end = len(nodes) - 1

#             edges.append((idx, body_start))   # enter loop
#             edges.append((body_end, idx))     # back edge

#             exit_idx = len(nodes)
#             edges.append((idx, exit_idx))     # exit loop

#         # ===== NORMAL WITH CHILDREN =====
#         elif block["children"]:
#             child_start = len(nodes)
#             build_cfg_from_tree(block["children"], nodes, edges)
#             edges.append((idx, child_start))

#         prev_idx = idx
#         i += 1


# # =========================
# # MAIN
# # =========================
# def build_cfg_from_code(code):
#     code = normalize_code(code)
#     functions = split_functions(code)

#     nodes = []
#     edges = []

#     for func in functions:
#         tree = split_blocks(func)
#         build_cfg_from_tree(tree, nodes, edges)

#     if len(nodes) == 0:
#         nodes = [{"type": "NORMAL", "text": code, "children": []}]

#     if len(edges) == 0:
#         edges = [(i, i + 1) for i in range(len(nodes) - 1)]
#         if not edges:
#             edges = [(0, 0)]

#     edges = list(set(edges))

#     # ===== FEATURES =====
#     features = np.array([extract_features(n) for n in nodes])

#     # ===== ADJ MATRIX =====
#     n = len(nodes)
#     adj = np.zeros((n, n), dtype=np.float32)

#     for i, j in edges:
#         if i < n and j < n:
#             adj[i][j] = 1.0

#     return features, adj

In [3]:
import re
import numpy as np

# =========================
# NORMALIZE
# =========================
def normalize_code(code):
    if not isinstance(code, str):
        return ""
    code = code.replace("\\r", "\n").replace("\r", "\n")
    code = re.sub(r'\s+', ' ', code)
    return code.strip()

# =========================
# SPLIT FUNCTIONS
# =========================
def split_functions(code):
    functions = []
    pattern = re.compile(r'(function|constructor|fallback|receive)\b.*?\{', re.DOTALL)
    for match in pattern.finditer(code):
        start = match.start()
        brace_count = 0
        for i in range(match.end() - 1, len(code)):
            if code[i] == "{":
                brace_count += 1
            elif code[i] == "}":
                brace_count -= 1
                if brace_count == 0:
                    functions.append(code[start:i + 1])
                    break
    return functions

# =========================
# SPLIT BLOCKS → TREE (ĐÃ CẢI THIỆN for/while)
# =========================
def split_blocks(function_code):
    code = normalize_code(function_code)
    
    # Mask ; bên trong for(...; ...; ...) và while(...) để split không bị vỡ
    def mask_semicolons(m):
        return m.group(0).replace(';', '§')
    
    code = re.sub(r'(\bfor\b|\bwhile\b)\s*\(([^)]*)\)', mask_semicolons, code, flags=re.IGNORECASE)
    
    tokens = re.split(r'(;|\{|\})', code)
    # Restore ;
    tokens = [t.replace('§', ';') for t in tokens]
    
    stmts = []
    current = ""
    for t in tokens:
        current += t
        if t in [";", "{", "}"]:
            stmts.append(current.strip())
            current = ""
    if current.strip():
        stmts.append(current.strip())

    stack = []
    root = []
    current_list = root

    def classify(stmt):
        s = stmt.lower()
        if re.search(r'\bif\b', s): return "IF"
        elif re.search(r'\belse\b', s): return "ELSE"
        elif re.search(r'\b(for|while)\b', s): return "LOOP"
        elif re.search(r'\breturn\b', s): return "RETURN"
        elif re.search(r'\btry\b', s): return "TRY"
        elif re.search(r'\bcatch\b', s): return "CATCH"
        elif "assembly" in s: return "ASSEMBLY"
        elif re.search(r'\b(break|continue)\b', s): return "JUMP"
        else: return "NORMAL"

    for stmt in stmts:
        stmt = stmt.strip()
        if not stmt: continue
        node = {"type": classify(stmt), "text": stmt, "children": []}
        if stmt.endswith("{"):
            current_list.append(node)
            stack.append(current_list)
            current_list = node["children"]
        elif stmt == "}":
            if stack:
                current_list = stack.pop()
        else:
            current_list.append(node)
    return root

# =========================
# EXTRACT FEATURES (giữ nguyên)
# =========================
def extract_features(block):
    text = block["text"].lower()
    t = block["type"]
    tokens = re.findall(r'\w+', text)
    unique_tokens = len(set(tokens))

    num_calls = len(re.findall(r'\b(call|delegatecall|callcode|staticcall)\b', text))
    num_transfer = len(re.findall(r'\b(transfer|send)\b', text))
    num_require = len(re.findall(r'\b(require|assert|revert|throw)\b', text))
    has_delegatecall = int("delegatecall" in text)
    has_selfdestruct = int("selfdestruct" in text or "suicide" in text)
    has_tx_origin = int("tx.origin" in text)
    has_block_timestamp = int("block.timestamp" in text or "now" in text)
    has_msg_sender = int("msg.sender" in text)
    has_msg_value = int("msg.value" in text)
    num_arith = len(re.findall(r'[\+\-\*/%]', text))
    num_compare = len(re.findall(r'(==|!=|>|<|>=|<=)', text))
    has_state_write = int(bool(re.search(r'\b\w+\s*=\s*(?!=)', text)))
    has_mapping_access = int(bool(re.search(r'\w+\s*\[.*?\]', text)))
    has_external_call = int(num_calls > 0 or num_transfer > 0)
    num_children = len(block["children"])

    has_assembly = int("assembly" in text)
    num_emits = len(re.findall(r'\bemit\b', text))
    has_payable = int("payable" in text)
    has_balance = len(re.findall(r'\.balance', text))
    has_block_number = int("block.number" in text)
    has_ecrecover = int("ecrecover" in text)
    has_keccak256 = int("keccak256" in text)
    has_sha256 = int("sha256" in text)
    has_storage = int("storage" in text)
    has_memory = int("memory" in text)
    has_calldata = int("calldata" in text)
    num_mapping_writes = len(re.findall(r'\w+\s*\[.*?\]\s*=', text))
    num_array_writes = len(re.findall(r'\w+\s*\[[^\]]+\]\s*=', text))
    has_msg_sender_check = int(bool(re.search(r'(require|assert|if).*?msg\.sender|msg\.sender.*?(require|assert|if|==|!=)', text)))
    has_reentrancy_risk = int(bool(re.search(r'(call|send|transfer|delegatecall|staticcall).*?=', text, re.DOTALL)))
    num_assign = len(re.findall(r'(?<!\=)\=(?!\=)', text))

    has_try = int("try" in text)
    has_catch = int("catch" in text)
    has_break = int("break" in text)
    has_continue = int("continue" in text)
    has_unchecked = int("unchecked" in text)
    has_approve = int("approve" in text)
    has_transfer_from = int("transferFrom" in text)
    has_only_owner = int("onlyowner" in text)
    num_low_level_calls = len(re.findall(r'\.(call|delegatecall|staticcall)', text))

    return np.array([
        int(t == "IF"), int(t == "ELSE"), int(t == "LOOP"), int(t == "RETURN"),
        int(t == "TRY"), int(t == "CATCH"), int(t == "ASSEMBLY"),
        int(t == "MERGE"), int(t == "JUMP"),
        int(num_calls > 0), int(num_transfer > 0), int(num_require > 0),
        has_delegatecall, has_selfdestruct, has_tx_origin, has_block_timestamp,
        has_msg_sender, has_msg_value,
        has_state_write, has_mapping_access, has_external_call,
        int(num_arith > 0), int(num_compare > 0),
        num_children, len(tokens), unique_tokens, len(text),
        num_calls, num_transfer, num_require, num_arith, num_compare,
        has_assembly, num_emits, has_payable, has_balance, has_block_number,
        has_ecrecover, has_keccak256, has_sha256, has_storage, has_memory,
        has_calldata, num_mapping_writes, num_array_writes,
        int(has_msg_sender_check), int(has_reentrancy_risk), num_assign,
        has_try, has_catch, has_break, has_continue, has_unchecked,
        has_approve, has_transfer_from, has_only_owner, num_low_level_calls
    ], dtype=np.float32)

# =========================
# BUILD CFG (ĐÃ SỬA HOÀN TOÀN)
# =========================
def build_cfg_from_tree(blocks, nodes, edges, break_target=None, continue_target=None):
    if not blocks:
        return None, None

    current_entry = None
    prev_exit = None
    i = 0
    while i < len(blocks):
        block = blocks[i]
        idx = len(nodes)
        nodes.append(block)

        if current_entry is None:
            current_entry = idx

        # Kết nối từ node trước (nếu có)
        if prev_exit is not None:
            edges.append((prev_exit, idx))

        node_type = block["type"]
        text_lower = block["text"].lower()

        # ====================== XỬ LÝ JUMP (break / continue) ======================
        is_break = node_type == "JUMP" and "break" in text_lower
        is_continue = node_type == "JUMP" and "continue" in text_lower

        if is_break and break_target is not None:
            edges.append((idx, break_target))
            block_exit = None
        elif is_continue and continue_target is not None:
            edges.append((idx, continue_target))
            block_exit = None

        # ====================== IF / TRY ======================
        elif node_type in ["IF", "TRY"]:
            true_entry, true_exit = build_cfg_from_tree(block["children"], nodes, edges, break_target, continue_target)
            true_has_body = true_entry is not None

            has_else = False
            false_entry = false_exit = None
            if i + 1 < len(blocks) and blocks[i + 1]["type"] in ["ELSE", "CATCH"]:
                has_else = True
                false_entry, false_exit = build_cfg_from_tree(blocks[i + 1]["children"], nodes, edges, break_target, continue_target)
                i += 1

            merge_idx = len(nodes)
            nodes.append({"type": "MERGE", "text": "merge point", "children": []})

            # true branch
            if true_has_body:
                edges.append((idx, true_entry))
                if true_exit is not None:
                    edges.append((true_exit, merge_idx))
            else:
                edges.append((idx, merge_idx))

            # false branch
            if has_else:
                if false_entry is not None:
                    edges.append((idx, false_entry))
                    if false_exit is not None:
                        edges.append((false_exit, merge_idx))
                else:
                    edges.append((idx, merge_idx))
            else:
                # KHÔNG CÓ ELSE → false trực tiếp đến merge
                edges.append((idx, merge_idx))

            block_exit = merge_idx

        # ====================== LOOP ======================
        elif node_type == "LOOP":
            loop_header = idx
            exit_merge = len(nodes)
            nodes.append({"type": "MERGE", "text": "loop exit", "children": []})

            body_entry, body_exit = build_cfg_from_tree(
                block["children"], nodes, edges,
                break_target=exit_merge,
                continue_target=loop_header
            )
            body_has_body = body_entry is not None

            if body_has_body:
                edges.append((idx, body_entry))
                if body_exit is not None:
                    edges.append((body_exit, idx))   # back edge

            edges.append((idx, exit_merge))   # luôn có thể thoát loop
            block_exit = exit_merge

        # ====================== BLOCK CÓ CHILDREN ======================
        elif block["children"]:
            child_entry, child_exit = build_cfg_from_tree(block["children"], nodes, edges, break_target, continue_target)
            child_has_body = child_entry is not None
            if child_has_body:
                edges.append((idx, child_entry))
                block_exit = child_exit
            else:
                block_exit = idx
        else:
            block_exit = idx

        # RETURN cũng dừng flow
        if node_type == "RETURN":
            block_exit = None

        prev_exit = block_exit
        i += 1

    return current_entry, prev_exit


# =========================
# MAIN
# =========================
def build_cfg_from_code(code):
    code = normalize_code(code)
    functions = split_functions(code)
    nodes = []
    edges = []

    for func in functions:
        if not func.strip(): continue
        start_idx = len(nodes)
        tree = split_blocks(func)
        _, func_exit = build_cfg_from_tree(tree, nodes, edges)

        # Xử lý function rỗng
        if func_exit is None and len(nodes) > start_idx:
            func_exit = len(nodes) - 1

        # Thêm EXIT node
        exit_idx = len(nodes)
        nodes.append({"type": "EXIT", "text": "function exit", "children": []})

        if func_exit is not None:
            if nodes[func_exit]["type"] != "RETURN":
                edges.append((func_exit, exit_idx))

        # Tất cả RETURN → EXIT
        for j in range(start_idx, exit_idx):
            if nodes[j]["type"] == "RETURN":
                edges.append((j, exit_idx))

    # Fallback
    if len(nodes) == 0:
        nodes = [{"type": "NORMAL", "text": code, "children": []}]

    edges = list(set(edges))   # remove duplicate

    # === FEATURES + ADJ MATRIX ===
    features = np.array([extract_features(n) for n in nodes])
    n = len(nodes)
    adj = np.zeros((n, n), dtype=np.float32)
    for i, j in edges:
        if i is not None and j is not None and 0 <= i < n and 0 <= j < n:
            adj[i][j] = 1.0

    return features, adj

In [4]:
# code = """
# contract Test {
#     function foo(uint x) public returns (uint) {
#         if (x > 10) {
#             x = x + 1;
#             return x;
#         } else {
#             x = x + 2;
#         }
#         x = x + 2;
#         sum=0;
#         while (x>0) {
#             sum+=x;
#             x-=1;
#         }
#         return x;
#     }
# }
# """

# features, edge_index = build_cfg_from_code(code)

# print("Nodes:", features.shape)
# print("Edges:", edge_index.shape)
# print(edge_index)

In [5]:
df = pd.read_csv(
    "DATASET/multilabel_BILSTM_BERT.csv",
)

label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                    'front_running', 'reentrancy', 'time_manipulation',
                    'unchecked_low_calls']
labels = (df[label_columns].values > 0).astype(int)

from tqdm import tqdm

node_counts = []
feature_list = []
edge_list = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    code = row["sourcecode"]

    features, edge_index = build_cfg_from_code(code)

    feature_list.append(features)
    edge_list.append(edge_index)

    num_nodes = features.shape[0]
    node_counts.append(num_nodes)

# ===== thống kê =====
node_counts = np.array(node_counts)

print("Mean nodes:", node_counts.mean())
print("Min nodes:", node_counts.min())
print("Max nodes:", node_counts.max())

100%|██████████| 45597/45597 [06:46<00:00, 112.17it/s]

Mean nodes: 188.82064609513785
Min nodes: 1
Max nodes: 3573


In [6]:
data = {
    "features": feature_list,
    "adj_matrices": edge_list,
    "labels": labels
}

with open("DATASET/gnn_input.pkl", "wb") as f:
    pickle.dump(data, f)

print("✅ Saved!")

✅ Saved!


In [7]:


# # load data
# with open("DATASET/gnn_input.pkl", "rb") as f:
#     data = pickle.load(f)

# feature_list = data["features"]

# # mỗi features = (num_nodes, num_features)
# num_nodes_list = [feat.shape[0] for feat in feature_list]

# avg_nodes = np.mean(num_nodes_list)
# max_nodes = np.max(num_nodes_list)
# min_nodes = np.min(num_nodes_list)

# print(f"Avg nodes: {avg_nodes:.2f}")
# print(f"Max nodes: {max_nodes}")
# print(f"Min nodes: {min_nodes}")
# print(len(num_nodes_list))

In [8]:
import torch
import pickle
import numpy as np
import torch.nn.functional as F

from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 57
num_classes = len(label_list[0])

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim), dtype=np.float32)
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features.astype(np.float32)

def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)
    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(label, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

data_list = [
    create_pyg_data(feat, adj, label, target_dim)
    for feat, adj, label in zip(feature_list, adj_matrices, label_list)
]

# nếu muốn shuffle trước khi split thì mở 2 dòng dưới
# indices = np.random.permutation(len(data_list))
# data_list = [data_list[i] for i in indices]

split_idx = int(len(data_list) * 0.8)
train_loader = DataLoader(data_list[:split_idx], batch_size=32, shuffle=True)
test_loader = DataLoader(data_list[split_idx:], batch_size=32, shuffle=False)


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:

# =========================
# MODEL
# =========================
# class GAT(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
#         super(GAT, self).__init__()
#         self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.3)
#         self.bn1 = BatchNorm1d(hidden_channels * heads)

#         self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=0.3)
#         self.bn2 = BatchNorm1d(hidden_channels)

#         self.lin = Linear(hidden_channels, out_channels)

#     def forward(self, data):
#         x, edge_index, batch = data.x, data.edge_index, data.batch

#         x = self.conv1(x, edge_index)
#         x = self.bn1(x)
#         x = F.relu(x)

#         x = self.conv2(x, edge_index)
#         x = self.bn2(x)
#         x = F.relu(x)

#         x = global_mean_pool(x, batch)
#         x = F.dropout(x, p=0.3, training=self.training)

#         x = self.lin(x)
#         x = torch.sigmoid(x)   # giữ sigmoid như bạn muốn
#         return x

# class GAT(torch.nn.Module):
#     def __init__(self, in_channels=57, hidden_channels=64, num_classes=8):
#         super().__init__()

#         # ===== 2 GAT layers =====
#         self.conv1 = GATConv(in_channels, hidden_channels, heads=4)
#         self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=2)

#         # ===== Feature layer (64) =====
#         self.fc1 = torch.nn.Linear(hidden_channels * 2, 64)

#         # ===== Output =====
#         self.lin = torch.nn.Linear(64, num_classes)

#         self.dropout = 0.3

#     def forward(self, data):
#         x, edge_index, batch = data.x, data.edge_index, data.batch

#         # ===== Layer 1 =====
#         x = self.conv1(x, edge_index)
#         x = F.relu(x)
#         x = F.dropout(x, p=self.dropout, training=self.training)

#         # ===== Layer 2 =====
#         x = self.conv2(x, edge_index)
#         x = F.relu(x)
#         x = F.dropout(x, p=self.dropout, training=self.training)

#         # ===== Pooling =====
#         x = global_mean_pool(x, batch)   # (B, hidden_channels)
#         x = F.dropout(x, p=self.dropout, training=self.training)

#         # ===== Feature 64 =====
#         x = self.fc1(x)
#         x = F.relu(x)   # (B, 64)
    
#         # ===== Output =====
#         x = self.lin(x)
#         x = torch.sigmoid(x)
#         return x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool


class GAT(nn.Module):
    def __init__(
        self,
        in_channels=57,
        hidden_channels=64,
        num_classes=8
    ):
        super(GAT, self).__init__()

        # ===== GAT Layers =====
        self.conv1 = GATConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=4,
            dropout=0.3
        )

        self.conv2 = GATConv(
            in_channels=hidden_channels * 4,
            out_channels=hidden_channels,
            heads=2,
            dropout=0.3
        )

        # ===== Batch Normalization =====
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels * 2)

        # ===== Dense Layers =====
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)

        # ===== Output Layer =====
        self.fc2 = nn.Linear(hidden_channels, num_classes)

        self.dropout = 0.3

    def forward(self, data):

        # ===== Inputs =====
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # ===== GAT Layer 1 =====
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== GAT Layer 2 =====
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Global Pooling =====
        x = global_mean_pool(x, batch)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Dense Layer =====
        x = self.fc1(x)
        x = F.relu(x)

        # ===== Output =====
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x

# =========================
# SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

gnn_model = GAT(in_channels=target_dim, hidden_channels=64).to(device)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

# =========================
# TRAIN / EVAL
# =========================
def train_one_epoch(model, loader, optimizer, criterion, device, threshold=0.5):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()
        out = model(data)   # out là xác suất vì đã sigmoid

        y = data.y
        if y.dim() == 1:
            y = y.view(-1, num_classes)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (out >= threshold).float()

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    precision = precision_score(all_labels, all_preds, average='micro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='micro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)

    return avg_loss, precision, recall, f1

In [10]:
from sklearn.metrics import classification_report

def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_outputs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data)   # xác suất
            y = data.y
            if y.dim() == 1:
                y = y.view(-1, num_classes)

            loss = criterion(out, y)
            total_loss += loss.item()

            preds = (out >= threshold).float()

            all_outputs.append(out.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    all_outputs = np.concatenate(all_outputs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0).astype(int)
    all_labels = np.concatenate(all_labels, axis=0).astype(int)

    report = classification_report(
        all_labels,
        all_preds,
        zero_division=0
    )

    return avg_loss, report, all_outputs, all_labels, all_preds

In [11]:

# =========================
# TRAIN LOOP
# =========================
num_epochs = 50
best_test_loss = float("inf")

for epoch in range(num_epochs):
    train_loss, train_precision, train_recall, train_f1 = train_one_epoch(
        gnn_model, train_loader, optimizer, criterion, device
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Precision: {train_precision:.4f} | "
        f"Recall: {train_recall:.4f} | "
        f"F1: {train_f1:.4f}"
    )

    if (epoch + 1) % 10 == 0:
        test_loss, report, gnn_test_outputs, y_test, y_pred = evaluate(
            gnn_model, test_loader, criterion, device
        )

        print("Test Loss:", test_loss)
        print(report)

    # nếu bạn muốn theo dõi luôn trên test mỗi epoch thì mở phần dưới
    # test_loss, test_precision, test_recall, test_f1, _, _ = evaluate(
    #     gnn_model, test_loader, criterion, device
    # )
    # print(
    #     f"                Test Loss: {test_loss:.4f} | "
    #     f"Precision: {test_precision:.4f} | "
    #     f"Recall: {test_recall:.4f} | "
    #     f"F1: {test_f1:.4f}"
    # )


Epoch 01/50 | Train Loss: 0.4408 | Precision: 0.7603 | Recall: 0.5924 | F1: 0.6659
Epoch 02/50 | Train Loss: 0.3913 | Precision: 0.7831 | Recall: 0.6512 | F1: 0.7111
Epoch 03/50 | Train Loss: 0.3756 | Precision: 0.7855 | Recall: 0.6744 | F1: 0.7257
Epoch 04/50 | Train Loss: 0.3633 | Precision: 0.7891 | Recall: 0.6847 | F1: 0.7332
Epoch 05/50 | Train Loss: 0.3565 | Precision: 0.7913 | Recall: 0.6927 | F1: 0.7387
Epoch 06/50 | Train Loss: 0.3499 | Precision: 0.7942 | Recall: 0.6999 | F1: 0.7440
Epoch 07/50 | Train Loss: 0.3451 | Precision: 0.7968 | Recall: 0.7091 | F1: 0.7504
Epoch 08/50 | Train Loss: 0.3437 | Precision: 0.7961 | Recall: 0.7116 | F1: 0.7515
Epoch 09/50 | Train Loss: 0.3408 | Precision: 0.7964 | Recall: 0.7176 | F1: 0.7549
Epoch 10/50 | Train Loss: 0.3362 | Precision: 0.7959 | Recall: 0.7245 | F1: 0.7585
Test Loss: 0.41021540489113123
              precision    recall  f1-score   support

           0       0.84      0.45      0.59      6091
           1       0.81      0

In [12]:
torch.save(gnn_model.state_dict(), "DATASET/gat.pth")

In [13]:
test_loss, report, gnn_test_outputs, y_test, y_pred = evaluate(
    gnn_model, test_loader, criterion, device
)

print("Test Loss:", test_loss)
print(report)

Test Loss: 0.3977229444604171
              precision    recall  f1-score   support

           0       0.87      0.37      0.52      6091
           1       0.85      0.15      0.25       476
           2       0.88      0.96      0.92      5755
           3       0.91      0.77      0.84      1268
           4       0.68      0.37      0.48      1018
           5       0.78      0.12      0.20      3798
           6       0.92      0.03      0.05       458
           7       0.96      0.25      0.40      4039

   micro avg       0.88      0.47      0.61     22903
   macro avg       0.86      0.38      0.46     22903
weighted avg       0.87      0.47      0.55     22903
 samples avg       0.62      0.49      0.53     22903

